# Lab 2 — Feature engineering for time series

**Goal:** turn a raw series \(y_t\) into a **supervised learning table** \((X_t, y_{t+1})\) or \((X_t, y_t)\) with **no leakage** — features at time \(t\) may only use information available **at or before** \(t\) (for one-step-ahead targets, typically information **strictly before** the predicted time).

**Prerequisite:** `data/synthetic_series.csv` from `generate_data.ipynb`.

Theory aligns with `time_series_forecasting.md` (feature engineering, ACF/differencing ideas, STL-based summaries, Fourier terms).

## 1. What counts as a feature?

Any numerical summary derived from the past can be a feature: **lags**, **rolling statistics**, **calendar dummies**, **harmonics (Fourier)**, **decomposition outputs** (trend/seasonal/remainder at \(t\)), **differences**, and global summaries (e.g. ACF-based statistics computed on a **training** window only — not on the full sample if you later evaluate on a test set).

**Leakage rule:** if the target is \(y_{t+1}\), features must not use \(y_{t+1}, y_{t+2}, \ldots\). For **real-time** forecasting at the end of the sample, you only have past values and known future calendars.

## 2. Lags, differences, seasonal differences

- **Lags:** \(y_{t-1}, y_{t-2}, \ldots\) capture short memory.
- **First difference:** \(\Delta y_t = y_t - y_{t-1}\) — approximates **change**; helps when the mean drifts.
- **Seasonal difference (lag \(m\)):** \(\Delta_m y_t = y_t - y_{t-m}\) — removes **seasonal level** when period is \(m\).

Autocorrelations of \(\Delta y_t\) or \(\Delta_m y_t\) are often used as **global** characterisations of the series (see ACF features in the notes).

## 3. Rolling statistics

Rolling mean / std / min / max over a window of past \(y\) summarise **local level** and **volatility**. They must be **causal**: at time \(t\), use only \(y_{t}, y_{t-1}, \ldots\) (or \(y_{t-1}\) and earlier if you predict \(y_{t+1}\) and want strict past-only features).

## 4. Calendar and Fourier features

**Calendar:** day-of-week, month, holidays — enter as integers with **one-hot** or **sin/cos** encoding to avoid arbitrary ordering.

**Fourier terms** approximate smooth seasonality with few parameters:

\[
s_{j,t} = \sin\left(\frac{2\pi j t}{m}\right), \quad c_{j,t} = \cos\left(\frac{2\pi j t}{m}\right), \quad j=1,\ldots, K
\]

with seasonal period \(m\). This is the basis of **dynamic harmonic regression** when combined with ARIMA errors (covered in Lab 4 context).

## 5. STL-based features

After **STL** (additive on the chosen scale), you can use **trend** and **seasonal** components as inputs, or derive **strength of trend / seasonality** (variance ratios — see notes). STL is **causal** only if fit on **past data**; fitting STL once on the full series and using in-sample components everywhere **leaks** test information — for practice, fit STL on the **train** segment only, or use **rolling** STL for production.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("data/synthetic_series.csv")
META_PATH = Path("data/synthetic_series_meta.json")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Run generate_data.ipynb first.")

raw = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
df = raw.set_index("date").asfreq("D")
if META_PATH.exists():
    with open(META_PATH, encoding="utf-8") as f:
        meta = json.load(f)
    SEASONAL_PERIOD = int(meta.get("seasonal_period", 7))
else:
    SEASONAL_PERIOD = 7

y = df["y"].astype(float)
exog = df[["temp", "promo"]].astype(float) if {"temp", "promo"}.issubset(df.columns) else None
print("n =", len(y), "  m =", SEASONAL_PERIOD)

In [ ]:
def add_calendar_features(idx: pd.DatetimeIndex) -> pd.DataFrame:
    out = pd.DataFrame(index=idx)
    out["dow"] = idx.dayofweek
    out["month"] = idx.month
    out["dayofyear"] = idx.dayofyear
    # Cyclical encoding (smooth; avoids arbitrary Mon=0)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    out["month_sin"] = np.sin(2 * np.pi * (out["month"] - 1) / 12)
    out["month_cos"] = np.cos(2 * np.pi * (out["month"] - 1) / 12)
    return out


def add_fourier_terms(idx: pd.DatetimeIndex, period: int, k: int = 3) -> pd.DataFrame:
    t = np.arange(len(idx), dtype=float)
    data = {}
    for j in range(1, k + 1):
        data[f"fourier_sin_{j}"] = np.sin(2 * np.pi * j * t / period)
        data[f"fourier_cos_{j}"] = np.cos(2 * np.pi * j * t / period)
    return pd.DataFrame(data, index=idx)


cal = add_calendar_features(y.index)
four = add_fourier_terms(y.index, SEASONAL_PERIOD, k=3)
cal.head()

In [ ]:
def add_lag_roll_features(s: pd.Series, max_lag: int, win: int) -> pd.DataFrame:
    out = pd.DataFrame(index=s.index)
    for ell in range(1, max_lag + 1):
        out[f"lag_{ell}"] = s.shift(ell)
    out["roll_mean"] = s.shift(1).rolling(win, min_periods=win).mean()
    out["roll_std"] = s.shift(1).rolling(win, min_periods=win).std()
    out["diff1"] = s.diff(1)
    out["diff_m"] = s.diff(SEASONAL_PERIOD)
    return out


MAX_LAG = 14
ROLL_WIN = SEASONAL_PERIOD * 2
lag_roll = add_lag_roll_features(y, MAX_LAG, ROLL_WIN)
lag_roll.head(20)

In [ ]:
from statsmodels.tsa.seasonal import STL

# Example: STL on full series for EDA only. For train/test modelling, refit STL on train.
stl = STL(y, seasonal=2 * SEASONAL_PERIOD + 1, robust=True)
res = stl.fit()
stl_feat = pd.DataFrame(
    {"stl_trend": res.trend, "stl_seasonal": res.seasonal, "stl_resid": res.resid},
    index=y.index,
)

y_sa = y - stl_feat["stl_seasonal"]  # seasonally adjusted (additive)
var_r = float(np.var(res.resid.dropna()))
ft = max(
    0.0,
    min(
        1.0,
        1.0 - var_r / max(np.var(y_sa.dropna() - res.trend.dropna()), 1e-12),
    ),
)  # strength of trend (approx.; notes use VAR formulations)
y_dt = y - res.trend
fs = max(
    0.0,
    min(1.0, 1.0 - var_r / max(np.var(y_dt.dropna()), 1e-12)),
)  # strength of seasonality (approx.)
print(f"Approx strength of trend: {ft:.3f}")
print(f"Approx strength of seasonality: {fs:.3f}")

In [ ]:
def build_supervised_table(target: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """Rows: features known at t (lags use y_{t-1}, ...); target is y_t. Drop NA from warm-up."""
    parts = [cal, four, lag_roll, stl_feat]
    if exog is not None:
        parts.append(exog.add_prefix("exog_"))
    X = pd.concat(parts, axis=1)
    tbl = X.copy()
    tbl["target_y"] = target
    tbl = tbl.dropna()
    return tbl.drop(columns=["target_y"]), tbl["target_y"]


X_all, y_all = build_supervised_table(y)
print("Feature matrix shape:", X_all.shape)
X_all.iloc[:3, :8]

## Interview checklist

1. **Causality** — every feature at \(t\) is computable when the forecast is made.
2. **Target horizon** — same pipeline, different shift for \(h\)-step targets.
3. **STL / scaling** — fit scalers and STL on **train** only, apply to test.
4. **Sparse events** — promo, holidays as dummies; interactions with seasonality if needed.

**Next:** Lab 3 — baselines and linear models on these features.